# Stage 05 — Feature Engineering

Feature engineering is the process of creating, transforming, selecting,
or removing features to improve the representation of data for a machine
learning model.

In this stage, we will investigate the Ames Housing dataset and determine
whether any feature-engineering operations are justified.

We will investigate:

1. Identifier features
2. Constant and near-constant features
3. Highly correlated features
4. Multicollinearity
5. Skewed numerical features
6. Domain-based feature creation
7. Feature transformations
8. Feature selection

No feature will be removed or created without a reason supported by
data analysis or domain understanding.

## Feature Engineering Principle

Feature engineering is not simply the process of removing columns.

A feature can be:

- Removed
- Transformed
- Combined with another feature
- Used to create a new feature
- Retained without modification

Every decision should have a reason.

We should be able to answer:

1. Why are we changing this feature?
2. What information does it contain?
3. What problem does the transformation solve?
4. Could the transformation introduce data leakage?
5. Does the model actually benefit from the change?

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

## 1. Load the Dataset

The feature-engineering analysis begins with the original dataset.

We will perform feature engineering using the predictor variables and
the target variable `SalePrice`.

The dataset used here is the same Ames Housing dataset used during the
earlier data-understanding and transformation stages.

In [2]:
DATA_PATH = "../artifacts/data_ingestion/AmesHousing.txt"

df = pd.read_csv(
    DATA_PATH,
    sep="\t"
)

print("Dataset shape:", df.shape)

Dataset shape: (2930, 82)


## 2. Separate Features and Target

The target variable is `SalePrice`.

The remaining columns represent the predictor variables.

We separate them because feature engineering will primarily be performed
on the predictor variables.

The target will be kept separate to avoid accidentally using it as an
input feature.

In [3]:
X = df.drop(
    columns=["SalePrice"]
)

y = df["SalePrice"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (2930, 81)
Target: (2930,)


## 3. Train/Test Split

Feature-engineering decisions must be developed without allowing the
test dataset to influence the modeling process.

We therefore maintain the same 80/20 split used during data
transformation.

The `random_state=42` ensures that the same observations are assigned to
the training and test sets as in the previous notebook.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (2344, 81)
X_test : (586, 81)


## 4. Identifier Features

An identifier is a value whose primary purpose is to identify an
observation rather than describe the underlying object.

Identifier-like columns can sometimes introduce noise because their
numeric values may have no meaningful relationship with the target.

We will investigate identifier candidates before deciding whether they
should be excluded.

In [5]:
identifier_candidates = [
    column
    for column in X_train.columns
    if (
        "id" in column.lower()
        or "pid" in column.lower()
        or "number" in column.lower()
    )
]

identifier_candidates

['PID']

In [6]:
for column in identifier_candidates:

    print("=" * 60)
    print(f"Feature: {column}")
    print("=" * 60)

    print("Data type:", X_train[column].dtype)
    print("Unique values:", X_train[column].nunique())
    print("Missing values:", X_train[column].isna().sum())
    print()

Feature: PID
Data type: int64
Unique values: 2344
Missing values: 0



In [7]:
identifier_summary = pd.DataFrame({
    "unique_values": [
        X_train[column].nunique()
        for column in identifier_candidates
    ],
    "rows": len(X_train)
}, index=identifier_candidates)

identifier_summary["unique_ratio"] = (
    identifier_summary["unique_values"]
    / identifier_summary["rows"]
)

identifier_summary

,unique_values,rows,unique_ratio
PID,2344,2344,1.0


In [8]:
for column in identifier_candidates:

    print(
        column,
        "duplicates:",
        X_train[column].duplicated().sum()
    )

PID duplicates: 0


### Identifier Investigation

Identifier-like columns were inspected for:

- Number of unique values
- Uniqueness ratio
- Duplicate values
- Missing values
- Semantic meaning

A feature will only be excluded if its role as an identifier is
confirmed and it does not represent meaningful predictive information.

We will preserve the original dataset until the feature-engineering
decisions have been finalized.

## 5. Constant and Near-Constant Features

A constant feature has the same value for every observation.

For example:

    Feature A
    ---------
    1
    1
    1
    1
    1

Such a feature provides no information for distinguishing observations.

A near-constant feature has one value occurring for almost every
observation.

These features may also provide little predictive information.

We will investigate them before deciding whether any should be removed.


In [9]:
constant_features = [
    column
    for column in X_train.columns
    if X_train[column].nunique(dropna=False) <= 1
]

print(
    "Constant features:",
    constant_features
)

Constant features: []


## 5.1 Near-Constant Features

A near-constant feature is dominated by one value.

We use a 99% frequency threshold as an exploratory criterion.

This is not an automatic deletion rule.

A near-constant feature may still contain useful information, especially
if the rare category has an important relationship with the target.

In [10]:
near_constant_features = []

threshold = 0.99

for column in X_train.columns:

    value_counts = (
        X_train[column]
        .value_counts(
            normalize=True,
            dropna=False
        )
    )

    if (
        len(value_counts) > 0
        and value_counts.iloc[0] >= threshold
    ):
        near_constant_features.append(column)

near_constant_features

['Street', 'Utilities', 'Condition 2', 'Pool Area', 'Pool QC']

In [11]:
for column in near_constant_features:

    print("=" * 60)
    print(column)
    print("=" * 60)

    print(
        X_train[column]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .head()
    )

    print()

Street
Street
Pave    0.995307
Grvl    0.004693
Name: proportion, dtype: float64

Utilities
Utilities
AllPub    0.998720
NoSewr    0.000853
NoSeWa    0.000427
Name: proportion, dtype: float64

Condition 2
Condition 2
Norm      0.991468
Feedr     0.004266
Artery    0.001280
PosA      0.000853
PosN      0.000853
Name: proportion, dtype: float64

Pool Area
Pool Area
0      0.994881
228    0.000427
648    0.000427
480    0.000427
555    0.000427
Name: proportion, dtype: float64

Pool QC
Pool QC
NaN    0.994881
Ex     0.001706
Gd     0.001706
TA     0.001280
Fa     0.000427
Name: proportion, dtype: float64



### Interpretation

Constant and near-constant features were investigated as potential
sources of unnecessary complexity.

Constant features contain no variation and therefore cannot help
distinguish observations.

Near-constant features require more careful consideration because the
small number of observations belonging to the rare category may still
contain predictive information.

No feature is removed solely because it meets the numerical threshold.

## 6. Correlation Analysis

Correlation measures the strength and direction of a linear relationship
between two numerical variables.

The Pearson correlation coefficient ranges from -1 to +1.

- +1 → perfect positive linear relationship
- 0  → no linear relationship
- -1 → perfect negative linear relationship

Correlation can help us identify relationships between numerical
features, but correlation alone should not be used as an automatic
feature-removal rule.

## 6.1 Feature-to-Target Correlation

We first examine the correlation between numerical predictor variables
and the target variable `SalePrice`.

This helps us understand which numerical variables have strong linear
relationships with house prices.

A strong correlation does not prove causation, and a weak correlation
does not necessarily mean that a feature is useless.

In [13]:
numerical_features = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

target_correlation = (
    X_train[numerical_features]
    .corrwith(y_train)
    .sort_values(
        ascending=False
    )
)

target_correlation

Overall Qual       0.795298
Gr Liv Area        0.698315
Garage Cars        0.644304
Garage Area        0.633106
Total Bsmt SF      0.612256
1st Flr SF         0.607433
Year Built         0.545409
Full Bath          0.542053
Year Remod/Add     0.517653
Garage Yr Blt      0.516211
Mas Vnr Area       0.490912
TotRms AbvGrd      0.475455
Fireplaces         0.467501
BsmtFin SF 1       0.423906
Wood Deck SF       0.333045
Lot Frontage       0.328726
Open Porch SF      0.297722
Bsmt Full Bath     0.286515
Half Bath          0.285369
2nd Flr SF         0.278977
Lot Area           0.261336
Bsmt Unf SF        0.163571
Bedroom AbvGr      0.149269
Screen Porch       0.136936
Pool Area          0.079020
3Ssn Porch         0.034845
Mo Sold            0.030714
BsmtFin SF 2       0.027205
Low Qual Fin SF   -0.016025
Misc Val          -0.017729
Order             -0.020621
Bsmt Half Bath    -0.023675
Yr Sold           -0.037686
MS SubClass       -0.066351
Overall Cond      -0.104085
Kitchen AbvGr     -0

In [14]:
top_target_correlations = (
    target_correlation
    .abs()
    .sort_values(
        ascending=False
    )
    .head(10)
)

top_target_correlations

Overall Qual      0.795298
Gr Liv Area       0.698315
Garage Cars       0.644304
Garage Area       0.633106
Total Bsmt SF     0.612256
1st Flr SF        0.607433
Year Built        0.545409
Full Bath         0.542053
Year Remod/Add    0.517653
Garage Yr Blt     0.516211
dtype: float64

In [15]:
fig = px.bar(
    x=top_target_correlations.values,
    y=top_target_correlations.index,
    orientation="h",
    title="Top Numerical Features by Absolute Correlation with SalePrice",
    labels={
        "x": "Absolute Correlation",
        "y": "Feature"
    }
)

fig.show()

## 6.2 Feature-to-Feature Correlation

We now investigate correlations between predictor variables.

This is different from feature-to-target correlation.

Highly correlated predictors may contain overlapping information and can
create multicollinearity, particularly for certain statistical models.

However, high correlation does not automatically mean that one of the
features should be removed.

In [16]:
feature_correlation = (
    X_train[numerical_features]
    .corr()
)

feature_correlation

,Order,PID,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,...,Garage Area,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
Order,1.000000,0.162298,0.025458,-0.011381,0.029692,-0.053964,-0.017230,-0.049789,-0.074495,-0.045710,...,-0.034316,-0.027584,0.033328,0.028249,-0.025903,0.007924,0.055222,-0.008229,0.136663,-0.975998
PID,0.162298,1.000000,0.004461,-0.088477,0.029190,-0.254492,0.098990,-0.336819,-0.143478,-0.233278,...,-0.207632,-0.066120,-0.059371,0.170004,-0.016513,-0.018409,0.002463,-0.009971,-0.043896,0.020373
MS SubClass,0.025458,0.004461,1.000000,-0.409800,-0.189798,0.060837,-0.065864,0.044954,0.054746,0.020931,...,-0.088739,-0.006276,0.001647,-0.023582,-0.040550,-0.054276,-0.004200,-0.032913,0.001898,-0.032299
Lot Frontage,-0.011381,-0.088477,-0.409800,1.000000,0.472802,0.191379,-0.078714,0.104217,0.067479,0.189991,...,0.346775,0.093893,0.160324,0.010830,0.049890,0.080330,0.199813,0.047264,0.006402,-0.000437
Lot Area,0.029692,0.029190,-0.189798,0.472802,1.000000,0.087235,-0.034115,0.021437,0.005470,0.115504,...,0.209233,0.144854,0.087039,0.026690,0.020722,0.061617,0.102046,0.074496,0.009317,-0.021706
Overall Qual,-0.053964,-0.254492,0.060837,0.191379,0.087235,1.000000,-0.093930,0.577688,0.549354,0.428606,...,0.555378,0.255925,0.281533,-0.140020,0.018301,0.054176,0.033582,0.007028,0.010113,-0.012391
Overall Cond,-0.017230,0.098990,-0.065864,-0.078714,-0.034115,-0.093930,1.000000,-0.379934,0.049327,-0.149740,...,-0.159409,0.005484,-0.074325,0.072785,0.038915,0.041948,-0.017797,0.036470,-0.001421,0.036661
Year Built,-0.049789,-0.336819,0.044954,0.104217,0.021437,0.577688,-0.379934,1.000000,0.593944,0.312021,...,0.462920,0.243430,0.175653,-0.378666,0.018484,-0.033189,0.003183,-0.011183,-0.004272,-0.013608
Year Remod/Add,-0.074495,-0.143478,0.054746,0.067479,0.005470,0.549354,0.049327,0.593944,1.000000,0.179319,...,0.369275,0.217806,0.205353,-0.219312,0.036120,-0.036652,-0.010441,-0.003779,0.004693,0.033432
Mas Vnr Area,-0.045710,-0.233278,0.020931,0.189991,0.115504,0.428606,-0.149740,0.312021,0.179319,1.000000,...,0.372021,0.155568,0.150543,-0.128023,0.023566,0.069072,0.004129,0.055750,-0.000769,-0.003345


In [17]:
fig = px.imshow(
    feature_correlation,
    text_auto=".2f",
    aspect="auto",
    title="Numerical Feature Correlation Matrix"
)

fig.show()

In [18]:
upper_triangle = (
    feature_correlation
    .where(
        np.triu(
            np.ones(
                feature_correlation.shape
            ),
            k=1
        ).astype(bool)
    )
)

In [19]:
upper_triangle.head()

,Order,PID,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,...,Garage Area,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
Order,NaN,0.162298,0.025458,-0.011381,0.029692,-0.053964,-0.017230,-0.049789,-0.074495,-0.045710,...,-0.034316,-0.027584,0.033328,0.028249,-0.025903,0.007924,0.055222,-0.008229,0.136663,-0.975998
PID,NaN,NaN,0.004461,-0.088477,0.029190,-0.254492,0.098990,-0.336819,-0.143478,-0.233278,...,-0.207632,-0.066120,-0.059371,0.170004,-0.016513,-0.018409,0.002463,-0.009971,-0.043896,0.020373
MS SubClass,NaN,NaN,NaN,-0.409800,-0.189798,0.060837,-0.065864,0.044954,0.054746,0.020931,...,-0.088739,-0.006276,0.001647,-0.023582,-0.040550,-0.054276,-0.004200,-0.032913,0.001898,-0.032299
Lot Frontage,NaN,NaN,NaN,NaN,0.472802,0.191379,-0.078714,0.104217,0.067479,0.189991,...,0.346775,0.093893,0.160324,0.010830,0.049890,0.080330,0.199813,0.047264,0.006402,-0.000437
Lot Area,NaN,NaN,NaN,NaN,NaN,0.087235,-0.034115,0.021437,0.005470,0.115504,...,0.209233,0.144854,0.087039,0.026690,0.020722,0.061617,0.102046,0.074496,0.009317,-0.021706


In [20]:
high_correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlation_pairs.columns = [
    "feature_1",
    "feature_2",
    "correlation"
]

high_correlation_pairs = (
    high_correlation_pairs[
        high_correlation_pairs["correlation"]
        .abs() >= 0.80
    ]
    .sort_values(
        "correlation",
        key=lambda x: x.abs(),
        ascending=False
    )
)

high_correlation_pairs

,feature_1,feature_2,correlation
37,Order,Yr Sold,-0.975998
1054,Garage Cars,Garage Area,0.883871
292,Year Built,Garage Yr Blt,0.824690
508,Total Bsmt SF,1st Flr SF,0.813946
670,Gr Liv Area,TotRms AbvGrd,0.806434


In [21]:
X_train[
    ["Order", "Yr Sold"]
].head(20)


,Order,Yr Sold
381,382,2009
834,835,2009
1898,1899,2007
678,679,2009
700,701,2009
2073,2074,2007
2443,2444,2006
2583,2584,2006
1915,1916,2007
879,880,2009


In [22]:
X_train[
    ["Order", "Yr Sold"]
].describe()

,Order,Yr Sold
count,2344.000000,2344.000000
mean,1464.298635,2007.794795
std,842.982263,1.315117
min,1.000000,2006.000000
25%,737.750000,2007.000000
50%,1466.000000,2008.000000
75%,2195.250000,2009.000000
max,2930.000000,2010.000000


In [23]:
X_train[
    ["Order", "Yr Sold"]
].nunique()


Order      2344
Yr Sold       5
dtype: int64

In [24]:
X_train[
    ["Garage Cars", "Garage Area"]
].describe()

,Garage Cars,Garage Area
count,2343.000000,2343.000000
mean,1.746906,469.078959
std,0.746445,212.432786
min,0.000000,0.000000
25%,1.000000,319.000000
50%,2.000000,476.000000
75%,2.000000,576.000000
max,4.000000,1488.000000


In [25]:
fig = px.scatter(
    X_train,
    x="Garage Cars",
    y="Garage Area",
    title="Garage Cars vs Garage Area"
)

fig.show()

In [26]:
X_train[
    ["Year Built", "Garage Yr Blt"]
].describe()

,Year Built,Garage Yr Blt
count,2344.000000,2222.00000
mean,1970.506826,1977.39829
std,30.341434,25.67911
min,1872.000000,1895.00000
25%,1953.000000,1960.00000
50%,1972.000000,1978.00000
75%,2000.000000,2001.00000
max,2010.000000,2207.00000


In [27]:
X_train[
    ["Year Built", "Garage Yr Blt"]
].corr()

,Year Built,Garage Yr Blt
Year Built,1.00000,0.82469
Garage Yr Blt,0.82469,1.00000


In [28]:
garage_year_difference = (
    X_train["Garage Yr Blt"]
    - X_train["Year Built"]
)

garage_year_difference.describe()

count    2222.000000
mean        5.423492
std        16.802003
min       -20.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       201.000000
dtype: float64

In [29]:
fig = px.scatter(
    X_train,
    x="Total Bsmt SF",
    y="1st Flr SF",
    title="Total Basement Area vs First Floor Area"
)

fig.show()


In [30]:
X_train[
    ["Total Bsmt SF", "1st Flr SF"]
].describe()

,Total Bsmt SF,1st Flr SF
count,2343.000000,2344.000000
mean,1047.022194,1154.814420
std,436.567117,385.114269
min,0.000000,334.000000
25%,784.000000,879.750000
50%,988.000000,1082.000000
75%,1288.000000,1378.000000
max,6110.000000,5095.000000


In [33]:
correlation_decisions = high_correlation_pairs.copy()

correlation_decisions["decision"] = ""
correlation_decisions["reason"] = ""

correlation_decisions

,feature_1,feature_2,correlation,decision,reason
37,Order,Yr Sold,-0.975998,,
1054,Garage Cars,Garage Area,0.883871,,
292,Year Built,Garage Yr Blt,0.824690,,
508,Total Bsmt SF,1st Flr SF,0.813946,,
670,Gr Liv Area,TotRms AbvGrd,0.806434,,


In [34]:
correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Order",
    "decision"
] = "REMOVE ORDER"

correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Order",
    "reason"
] = (
    "Order behaves as an ordering/identifier variable rather than "
    "a meaningful property characteristic."
)

In [35]:
correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Garage Cars",
    "decision"
] = "KEEP BOTH"

correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Garage Cars",
    "reason"
] = (
    "Garage Cars represents capacity while Garage Area represents "
    "physical garage size."
)

In [36]:
correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Year Built",
    "decision"
] = "KEEP BOTH"

correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Year Built",
    "reason"
] = (
    "Year Built describes the house while Garage Yr Blt describes "
    "the garage construction year."
)

In [37]:
correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Total Bsmt SF",
    "decision"
] = "KEEP BOTH"

correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Total Bsmt SF",
    "reason"
] = (
    "Basement area and first-floor area represent different "
    "physical spaces."
)

In [38]:
correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Gr Liv Area",
    "decision"
] = "KEEP BOTH"

correlation_decisions.loc[
    correlation_decisions["feature_1"] == "Gr Liv Area",
    "reason"
] = (
    "Living area measures space while room count measures the "
    "number of rooms."
)

### Correlation Analysis Conclusion

Five highly correlated numerical feature pairs were identified.

The `Order` feature is treated differently because it behaves as an
ordering/identifier variable rather than a meaningful property
characteristic.

The remaining highly correlated pairs represent different aspects of
the property:

- Garage Cars and Garage Area represent garage capacity and size.
- Year Built and Garage Yr Blt represent house and garage construction
  years.
- Total Bsmt SF and 1st Flr SF represent different areas of the house.
- Gr Liv Area and TotRms AbvGrd represent living area and room count.

Therefore, high correlation alone is not considered sufficient evidence
to remove these predictive features.

Further multicollinearity analysis and model evaluation will be used
before making final feature-selection decisions.

### Correlation Analysis Conclusion

Five highly correlated numerical feature pairs were identified.

The `Order` feature is treated differently because it behaves as an
ordering/identifier variable rather than a meaningful property
characteristic.

The remaining highly correlated pairs represent different aspects of
the property:

- Garage Cars and Garage Area represent garage capacity and size.
- Year Built and Garage Yr Blt represent house and garage construction
  years.
- Total Bsmt SF and 1st Flr SF represent different areas of the house.
- Gr Liv Area and TotRms AbvGrd represent living area and room count.

Therefore, high correlation alone is not considered sufficient evidence
to remove these predictive features.

Further multicollinearity analysis and model evaluation will be used
before making final feature-selection decisions.

In [39]:
import statsmodels
print(statsmodels.__version__)

0.14.6


In [40]:
numerical_features = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

print(
    "Number of numerical features:",
    len(numerical_features)
)

Number of numerical features: 38


In [41]:
vif_features = [
    feature
    for feature in numerical_features
    if feature != "Order"
]
print(
    "Numerical features used for VIF:",
    len(vif_features)
)

Numerical features used for VIF: 37


In [42]:
vif_data = X_train[vif_features].copy()
vif_data = vif_data.fillna(
    vif_data.median()
)
print(
    "Missing values:",
    vif_data.isna().sum().sum()
)

Missing values: 0


In [43]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [44]:
vif_results = pd.DataFrame()

vif_results["feature"] = (
    vif_data.columns
)

vif_results["VIF"] = [
    variance_inflation_factor(
        vif_data.values,
        i
    )
    for i in range(
        vif_data.shape[1]
    )
]

vif_results = (
    vif_results
    .sort_values(
        "VIF",
        ascending=False
    )
)

vif_results

e:\Project_Works\ML_Learn\Ames Housing Prices\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
16,Gr Liv Area,inf
13,1st Flr SF,inf
14,2nd Flr SF,inf
15,Low Qual Fin SF,inf
12,Total Bsmt SF,1.333727e+05
11,Bsmt Unf SF,5.137847e+04
9,BsmtFin SF 1,4.151185e+04
36,Yr Sold,2.304311e+04
7,Year Remod/Add,2.149627e+04
6,Year Built,2.015245e+04


In [45]:
top_vif = (
    vif_results
    .head(15)
    .sort_values(
        "VIF"
    )
)

fig = px.bar(
    top_vif,
    x="VIF",
    y="feature",
    orientation="h",
    title="Highest VIF Numerical Features"
)

fig.show()

In [46]:
vif_results[
    vif_results["feature"].isin([
        "Garage Cars",
        "Garage Area",
        "Year Built",
        "Garage Yr Blt",
        "Total Bsmt SF",
        "1st Flr SF",
        "Gr Liv Area",
        "TotRms AbvGrd"
    ])
]

,feature,VIF
16,Gr Liv Area,inf
13,1st Flr SF,inf
12,Total Bsmt SF,1.333727e+05
6,Year Built,2.015245e+04
25,Garage Yr Blt,1.995944e+04
23,TotRms AbvGrd,7.957404e+01
26,Garage Cars,3.631722e+01
27,Garage Area,3.198434e+01


In [47]:
skewness = (
    X_train[numerical_features]
    .skew()
    .sort_values(
        ascending=False
    )
)

skewness

Misc Val           20.416138
Pool Area          15.898588
Lot Area           13.743796
Low Qual Fin SF    12.959105
3Ssn Porch         12.140302
Kitchen AbvGr       4.493078
BsmtFin SF 2        4.185302
Bsmt Half Bath      4.003080
Enclosed Porch      3.993799
Screen Porch        3.894896
Open Porch SF       2.664539
Mas Vnr Area        2.497916
Lot Frontage        1.735006
Wood Deck SF        1.564691
1st Flr SF          1.542293
BsmtFin SF 1        1.529633
MS SubClass         1.364770
Total Bsmt SF       1.343278
Gr Liv Area         1.303483
Bsmt Unf SF         0.910201
2nd Flr SF          0.875991
Fireplaces          0.771836
TotRms AbvGrd       0.767736
Half Bath           0.729985
Bsmt Full Bath      0.628831
Overall Cond        0.595367
Bedroom AbvGr       0.315893
Garage Area         0.292990
Mo Sold             0.211601
Overall Qual        0.204464
Full Bath           0.184097
Yr Sold             0.126187
PID                 0.057310
Order               0.002033
Garage Cars   

In [48]:
most_right_skewed = (
    skewness
    .head(15)
)

most_right_skewed

Misc Val           20.416138
Pool Area          15.898588
Lot Area           13.743796
Low Qual Fin SF    12.959105
3Ssn Porch         12.140302
Kitchen AbvGr       4.493078
BsmtFin SF 2        4.185302
Bsmt Half Bath      4.003080
Enclosed Porch      3.993799
Screen Porch        3.894896
Open Porch SF       2.664539
Mas Vnr Area        2.497916
Lot Frontage        1.735006
Wood Deck SF        1.564691
1st Flr SF          1.542293
dtype: float64

In [49]:
fig = px.bar(
    most_right_skewed.sort_values(),
    orientation="h",
    title="Most Right-Skewed Numerical Features",
    labels={
        "value": "Skewness",
        "index": "Feature"
    }
)

fig.show()

In [50]:
most_left_skewed = (
    skewness
    .sort_values()
    .head(15)
)

most_left_skewed

Year Built       -0.594285
Year Remod/Add   -0.428230
Garage Yr Blt    -0.303852
Garage Cars      -0.211890
Order             0.002033
PID               0.057310
Yr Sold           0.126187
Full Bath         0.184097
Overall Qual      0.204464
Mo Sold           0.211601
Garage Area       0.292990
Bedroom AbvGr     0.315893
Overall Cond      0.595367
Bsmt Full Bath    0.628831
Half Bath         0.729985
dtype: float64

In [51]:
fig = px.bar(
    most_left_skewed.sort_values(ascending=False),
    orientation="h",
    title="Most Left-Skewed Numerical Features",
    labels={
        "value": "Skewness",
        "index": "Feature"
    }
)

fig.show()

In [52]:
feature = "Lot Area"

print(
    X_train[feature].describe()
)

count      2344.000000
mean      10127.857509
std        8050.908132
min        1300.000000
25%        7466.500000
50%        9356.500000
75%       11484.250000
max      215245.000000
Name: Lot Area, dtype: float64


In [53]:
skew_threshold = 1.0

skewed_features = (
    skewness[
        skewness.abs() > skew_threshold
    ]
)

skewed_features

Misc Val           20.416138
Pool Area          15.898588
Lot Area           13.743796
Low Qual Fin SF    12.959105
3Ssn Porch         12.140302
Kitchen AbvGr       4.493078
BsmtFin SF 2        4.185302
Bsmt Half Bath      4.003080
Enclosed Porch      3.993799
Screen Porch        3.894896
Open Porch SF       2.664539
Mas Vnr Area        2.497916
Lot Frontage        1.735006
Wood Deck SF        1.564691
1st Flr SF          1.542293
BsmtFin SF 1        1.529633
MS SubClass         1.364770
Total Bsmt SF       1.343278
Gr Liv Area         1.303483
dtype: float64

In [54]:
feature = skewed_features.index[0]

print("Feature:", feature)
print(
    "Skewness:",
    skewness[feature]
)

Feature: Misc Val
Skewness: 20.416137552406934


In [55]:
fig = px.histogram(
    X_train,
    x=feature,
    nbins=50,
    title=f"Distribution of {feature}"
)

fig.show()

In [56]:
original = X_train[feature].copy()

transformed = np.log1p(original)

In [57]:
print(
    "Original skewness:",
    original.skew()
)

print(
    "Transformed skewness:",
    transformed.skew()
)

Original skewness: 20.416137552406934
Transformed skewness: 4.953296618749905


### Skewness Analysis Conclusion

Numerical features were analyzed for distributional asymmetry.

Highly skewed features were identified as candidates for further
investigation rather than automatic transformation.

For selected non-negative features, a log transformation can be tested
to determine whether it reduces extreme skewness.

A transformation will only be included in the final preprocessing
pipeline if it is meaningful for the feature and provides evidence of
improved model behavior.

Skewness alone is not a reason to remove a feature.

## 9. Domain-Based Feature Creation

Feature engineering can use domain knowledge to create variables that
represent meaningful concepts more directly.

For the Ames Housing dataset, one potentially useful feature is the age
of the house when it was sold.

House Age at Sale can be calculated as:

    HouseAgeAtSale = Yr Sold - Year Built

This represents how old the house was at the time of the transaction.

In [58]:
required_columns = [
    "Yr Sold",
    "Year Built"
]

for column in required_columns:
    print(
        column,
        "exists:",
        column in X_train.columns
    )

Yr Sold exists: True
Year Built exists: True


In [59]:
house_age_at_sale = (
    X_train["Yr Sold"]
    - X_train["Year Built"]
)
house_age_at_sale.describe()

count    2344.000000
mean       37.287969
std        30.387795
min        -1.000000
25%         8.000000
50%        35.500000
75%        55.000000
max       136.000000
dtype: float64

In [62]:
X_train.loc[
    house_age_at_sale < 0,
    [
        "Year Built",
        "Yr Sold",
        
    ]
]

,Year Built,Yr Sold
2180,2008,2007


In [63]:
X_train.loc[
    2180
]

Order                  2181
PID               908154195
MS SubClass              20
MS Zoning                RL
Lot Frontage          128.0
                    ...    
Misc Val              17000
Mo Sold                  10
Yr Sold                2007
Sale Type               New
Sale Condition      Partial
Name: 2180, Length: 81, dtype: object

In [64]:
print(
    "Negative house ages:",
    (house_age_at_sale < 0).sum()
)

Negative house ages: 1


### 9.1 Years Since Remodeling

The dataset contains the year the property was originally built,
the year it was last remodeled, and the year it was sold.

We can derive a new feature:

    YearsSinceRemodel = Yr Sold - Year Remod/Add

This represents the approximate number of years since the most recent
remodeling at the time of sale.

The feature will be validated for impossible values before we decide
whether to include it in the final feature set.

In [65]:
years_since_remodel = (
    X_train["Yr Sold"]
    - X_train["Year Remod/Add"]
)

In [66]:
years_since_remodel.describe()

count    2344.000000
mean       23.871160
std        20.783923
min        -2.000000
25%         5.000000
50%        16.000000
75%        43.000000
max        60.000000
dtype: float64

In [67]:
(years_since_remodel < 0).sum()

np.int64(2)

In [70]:
X_train.loc[
    years_since_remodel < 0,
    [
        "Year Built",
        "Year Remod/Add",
        "Yr Sold"
    ]
]

,Year Built,Year Remod/Add,Yr Sold
1702,2007,2008,2007
2180,2008,2009,2007


### Data Quality Observation — YearsSinceRemodel

The derived `YearsSinceRemodel` feature produced two negative values.

These records contain inconsistent year information where `Year Remod/Add`
is later than `Yr Sold`.

The formula used to calculate the feature is valid, but the underlying
records are inconsistent.

We will not manually modify the original year values because the correct
values cannot be determined from the available information.

The anomaly will be documented and considered during the final
preprocessing and feature-selection stages.

In [71]:
years_since_remodel.describe()

count    2344.000000
mean       23.871160
std        20.783923
min        -2.000000
25%         5.000000
50%        16.000000
75%        43.000000
max        60.000000
dtype: float64

In [72]:
print(
    "Skewness:",
    years_since_remodel.skew()
)

Skewness: 0.4272190638433843


In [73]:
print(
    "Correlation with SalePrice:",
    years_since_remodel.corr(y_train)
)

Correlation with SalePrice: -0.5200964472924225


In [74]:
X_train[
    [
        "Garage Type",
        "Garage Yr Blt",
        "Garage Cars",
        "Garage Area"
    ]
].isna().sum()

Garage Type      120
Garage Yr Blt    122
Garage Cars        1
Garage Area        1
dtype: int64

In [75]:
X_train[
    [
        "Garage Type",
        "Garage Yr Blt",
        "Garage Cars",
        "Garage Area"
    ]
].head()

,Garage Type,Garage Yr Blt,Garage Cars,Garage Area
381,Attchd,1976.0,2.0,479.0
834,Attchd,1967.0,2.0,538.0
1898,CarPort,1962.0,2.0,462.0
678,Detchd,1956.0,2.0,420.0
700,NaN,NaN,0.0,0.0


In [76]:
has_garage = (
    X_train["Garage Area"]
    .notna()
    .astype(int)
)

In [77]:
has_garage.value_counts()

Garage Area
1    2343
0       1
Name: count, dtype: int64

In [78]:
garage_age = (
    X_train["Yr Sold"]
    - X_train["Garage Yr Blt"]
)

In [79]:
garage_age.describe()

count    2222.000000
mean       30.393789
std        25.726249
min      -200.000000
25%         7.000000
50%        29.000000
75%        48.000000
max       114.000000
dtype: float64

In [80]:
(garage_age < 0).sum()

np.int64(2)

In [81]:
X_train.loc[
    garage_age < 0,
    [
        "Garage Yr Blt",
        "Yr Sold"
    ]
]

,Garage Yr Blt,Yr Sold
2260,2207.0,2007
2180,2008.0,2007


In [82]:
pd.DataFrame({
    "HasGarage": has_garage,
    "GarageAge": garage_age
}).head(20)

,HasGarage,GarageAge
381,1,33.0
834,1,42.0
1898,1,45.0
678,1,53.0
700,1,NaN
2073,1,19.0
2443,1,8.0
2583,1,52.0
1915,1,42.0
879,1,2.0


In [83]:
garage_price_analysis = pd.DataFrame({
    "HasGarage": has_garage,
    "SalePrice": y_train
})

garage_price_analysis.groupby(
    "HasGarage"
)["SalePrice"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
HasGarage,,,
0,1,150909.000000,150909.0
1,2343,178594.018779,160000.0


In [84]:
area_columns = [
    "Total Bsmt SF",
    "1st Flr SF",
    "2nd Flr SF"
]

X_train[area_columns].describe()

,Total Bsmt SF,1st Flr SF,2nd Flr SF
count,2343.000000,2344.000000,2344.000000
mean,1047.022194,1154.814420,333.688140
std,436.567117,385.114269,427.141191
min,0.000000,334.000000,0.000000
25%,784.000000,879.750000,0.000000
50%,988.000000,1082.000000,0.000000
75%,1288.000000,1378.000000,701.000000
max,6110.000000,5095.000000,2065.000000


In [85]:
X_train[area_columns].isna().sum()

Total Bsmt SF    1
1st Flr SF       0
2nd Flr SF       0
dtype: int64

In [86]:
X_train.loc[
    X_train[area_columns].isna().any(axis=1),
    area_columns
].head(10)

,Total Bsmt SF,1st Flr SF,2nd Flr SF
1341,NaN,896,0


In [87]:
total_area = (
    X_train["Total Bsmt SF"].fillna(0)
    + X_train["1st Flr SF"].fillna(0)
    + X_train["2nd Flr SF"].fillna(0)
)

In [88]:
total_area.describe()


count     2344.000000
mean      2535.078072
std        798.307325
min        334.000000
25%       1993.500000
50%       2443.000000
75%       2972.000000
max      11752.000000
dtype: float64

In [89]:
total_area_df = pd.DataFrame({
    "TotalArea": total_area,
    "SalePrice": y_train
})

fig = px.scatter(
    total_area_df,
    x="TotalArea",
    y="SalePrice",
    title="Total Area vs SalePrice"
)

fig.show()

In [91]:
skewed_features

Misc Val           20.416138
Pool Area          15.898588
Lot Area           13.743796
Low Qual Fin SF    12.959105
3Ssn Porch         12.140302
Kitchen AbvGr       4.493078
BsmtFin SF 2        4.185302
Bsmt Half Bath      4.003080
Enclosed Porch      3.993799
Screen Porch        3.894896
Open Porch SF       2.664539
Mas Vnr Area        2.497916
Lot Frontage        1.735006
Wood Deck SF        1.564691
1st Flr SF          1.542293
BsmtFin SF 1        1.529633
MS SubClass         1.364770
Total Bsmt SF       1.343278
Gr Liv Area         1.303483
dtype: float64

In [92]:
X_train[
    skewed_features.index
].min().sort_values()

Misc Val              0.0
Pool Area             0.0
Low Qual Fin SF       0.0
3Ssn Porch            0.0
BsmtFin SF 2          0.0
Kitchen AbvGr         0.0
Bsmt Half Bath        0.0
Enclosed Porch        0.0
Wood Deck SF          0.0
Screen Porch          0.0
Open Porch SF         0.0
Mas Vnr Area          0.0
BsmtFin SF 1          0.0
Total Bsmt SF         0.0
MS SubClass          20.0
Lot Frontage         21.0
Gr Liv Area         334.0
1st Flr SF          334.0
Lot Area           1300.0
dtype: float64

In [93]:
non_negative_skewed = [
    feature
    for feature in skewed_features.index
    if X_train[feature].min() >= 0
]

non_negative_skewed

['Misc Val',
 'Pool Area',
 'Lot Area',
 'Low Qual Fin SF',
 '3Ssn Porch',
 'Kitchen AbvGr',
 'BsmtFin SF 2',
 'Bsmt Half Bath',
 'Enclosed Porch',
 'Screen Porch',
 'Open Porch SF',
 'Mas Vnr Area',
 'Lot Frontage',
 'Wood Deck SF',
 '1st Flr SF',
 'BsmtFin SF 1',
 'MS SubClass',
 'Total Bsmt SF',
 'Gr Liv Area']

In [94]:
feature = (
    skewness[
        non_negative_skewed
    ]
    .abs()
    .sort_values(
        ascending=False
    )
    .index[0]
)

print("Selected feature:", feature)
print(
    "Original skewness:",
    skewness[feature]
)

Selected feature: Misc Val
Original skewness: 20.416137552406934


In [95]:
original = X_train[feature].copy()

print(
    original.describe()
)

count     2344.000000
mean        58.055034
std        623.375121
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      17000.000000
Name: Misc Val, dtype: float64


In [96]:
fig = px.histogram(
    x=original,
    nbins=50,
    title=f"Original Distribution — {feature}"
)

fig.show()

In [97]:
original_skewness = original.skew()

print(
    "Original skewness:",
    original_skewness
)

Original skewness: 20.416137552406934


In [98]:
original_skewness = original.skew()

print(
    "Original skewness:",
    original_skewness
)

Original skewness: 20.416137552406934


In [99]:
print(
    transformed.describe()
)

count    2344.000000
mean        0.258791
std         1.301586
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         9.741027
Name: Misc Val, dtype: float64


In [100]:
transformed_skewness = transformed.skew()

print(
    "Transformed skewness:",
    transformed_skewness
)

Transformed skewness: 4.953296618749905


In [101]:
fig = px.histogram(
    x=original,
    nbins=50,
    title=f"Original — {feature}"
)

fig.show()

In [102]:
fig = px.histogram(
    x=transformed,
    nbins=50,
    title=f"Log1p Transformed — {feature}"
)

fig.show()

# Feature Engineering — Summary

This notebook investigated the major feature-engineering concepts required
for the Ames Housing prediction project.

The analysis covered:

- Identifier and non-predictive feature investigation
- Constant and near-constant features
- Target correlation
- Feature-to-feature correlation
- Multicollinearity and VIF
- Numerical feature skewness
- Domain-based feature creation
- Feature transformations
- Missing-value meaning and indicator features

Important conclusions:

1. A high correlation does not automatically mean a feature should be removed.
2. VIF is a diagnostic for multicollinearity, not an automatic deletion rule.
3. Skewness does not automatically require transformation.
4. Feature creation should be based on domain meaning.
5. Missing values can sometimes contain meaningful information.
6. Feature decisions should ultimately be validated using model performance.
7. The final preprocessing pipeline will apply all selected transformations
   consistently to training and unseen data.

The candidate engineered features identified during exploration will be
evaluated during the modeling stage rather than being permanently added
based solely on exploratory statistics.